In [ ]:
#TASK1 PROBLEM1
!pip install tensorflow numpy pandas matplotlib --quiet

import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import random
import os

print("=" * 50)
print("        PACKAGE VERSIONS")
print("=" * 50)
print(f"TensorFlow  : {tf.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"Pandas      : {pd.__version__}")
print(f"Matplotlib  : {matplotlib.__version__}")
print(f"Python OS   : {os.name}")
print("=" * 50)

gpus = tf.config.list_physical_devices('GPU')

print("\n GPU AVAILABILITY CHECK")
print("=" * 50)

if gpus:
    print(f"GPU detected: {len(gpus)} device(s)")
    for gpu in gpus:
        print(f"   → {gpu.name}")
else:
    print(" No GPU detected. Running on CPU.")
    print()
    print("   ℹ️  See comments in code for why CPU is slower")
    print("   ℹ️  and what changes a GPU setup would require.")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

os.environ['PYTHONHASHSEED'] = str(SEED)

print("\n RANDOM SEEDS")
print("=" * 50)
print(f" random.seed        → {SEED}  (Python built-in randomness)")
print(f" np.random.seed     → {SEED}  (NumPy ops & sklearn splits)")
print(f" tf.random.set_seed → {SEED}  (TF weights, dropout, augment)")
print(f" PYTHONHASHSEED     → {SEED}  (Python hash randomization)")
print()
print(" Reproducibility locked. Same results every run.")
print("=" * 50)
print("\n SETUP SUMMARY")
print("=" * 50)
print(f"  Framework  : TensorFlow/Keras {tf.__version__}")
print(f"  GPU        : {'Yes ' if gpus else 'No (CPU mode)'}")
print(f"  Random Seed: {SEED} (all sources seeded)")
print(f"  Status     : Ready for CNN training ")
print("=" * 50)

In [ ]:
#TASK1 PROBLEM2 : Dataset Exploration

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tensorflow.keras.datasets import mnist, cifar10
import random, os

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

(mnist_x_train, mnist_y_train), (mnist_x_test, mnist_y_test) = mnist.load_data()
(cifar_x_train, cifar_y_train), (cifar_x_test, cifar_y_test) = cifar10.load_data()

CIFAR10_CLASSES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

print("=" * 55)
print("  (a) ARRAY SHAPES")
print("=" * 55)

print("\n── MNIST ──────────────────────────────────────────")
print(f"  Train images : {mnist_x_train.shape}")
print(f"               → {mnist_x_train.shape[0]} samples, "
      f"{mnist_x_train.shape[1]}×{mnist_x_train.shape[2]} px, grayscale")
print(f"  Train labels : {mnist_y_train.shape}")
print(f"  Test  images : {mnist_x_test.shape}")
print(f"  Test  labels : {mnist_y_test.shape}")

print("\n── CIFAR-10 ────────────────────────────────────────")
print(f"  Train images : {cifar_x_train.shape}")
print(f"               → {cifar_x_train.shape[0]} samples, "
      f"{cifar_x_train.shape[1]}×{cifar_x_train.shape[2]} px, "
      f"{cifar_x_train.shape[3]} channels (RGB)")
print(f"  Train labels : {cifar_y_train.shape}")
print(f"  Test  images : {cifar_x_test.shape}")
print(f"  Test  labels : {cifar_y_test.shape}")

print("\n" + "=" * 55)
print("  (b) DATA TYPE & PIXEL VALUE RANGE")
print("=" * 55)

print("\n── MNIST ──────────────────────────────────────────")
print(f"  dtype        : {mnist_x_train.dtype}")
print(f"  Min value    : {mnist_x_train.min()}")
print(f"  Max value    : {mnist_x_train.max()}")
print(f"  Range        : [0, 255]  → 8-bit unsigned integers")

print("\n── CIFAR-10 ────────────────────────────────────────")
print(f"  dtype        : {cifar_x_train.dtype}")
print(f"  Min value    : {cifar_x_train.min()}")
print(f"  Max value    : {cifar_x_train.max()}")
print(f"  Range        : [0, 255]  → 8-bit unsigned integers")

print("\n Both datasets are raw uint8 pixel arrays.")
print("  For training, we will normalize to [0.0, 1.0]")
print("      by dividing by 255.0  (done in preprocessing step).")
print("\n" + "=" * 55)
print("  (c) MNIST TRAINING SET — SAMPLES PER CLASS")
print("=" * 55)

unique_classes, counts = np.unique(mnist_y_train, return_counts=True)
total = mnist_y_train.shape[0]

print(f"\n  {'Digit':<8} {'Count':>7}   {'% of total':>10}   Bar")
print("  " + "-" * 50)

for cls, cnt in zip(unique_classes, counts):
    pct    = cnt / total * 100
    bar    = "█" * int(pct / 1.5)          # scaled bar
    print(f"  {cls:<8} {cnt:>7,}   {pct:>9.2f}%   {bar}")

print("  " + "-" * 50)
print(f"  {'TOTAL':<8} {total:>7,}   {'100.00%':>10}")

std_dev = np.std(counts)
max_diff = counts.max() - counts.min()
print(f"\n  Std deviation across classes : {std_dev:.1f}")
print(f"  Max − Min count              : {max_diff:,}")

if std_dev < 300:
    print("\n BALANCED — class counts are very close.")
    print("     No over/under-sampling needed.")
else:
    print("\n  IMBALANCED — consider resampling strategies.")

mnist_indices = np.random.choice(len(mnist_x_train), 10, replace=False)
cifar_indices = np.random.choice(len(cifar_x_train), 10, replace=False)

fig = plt.figure(figsize=(18, 5))
fig.patch.set_facecolor('#0f0f0f')

fig.suptitle(
    "Dataset Samples: MNIST  (top)   |   CIFAR-10  (bottom)",
    fontsize=15, fontweight='bold', color='white', y=1.01
)

gs = gridspec.GridSpec(
    2, 10,
    figure=fig,
    hspace=0.55,
    wspace=0.15
)

for col, idx in enumerate(mnist_indices):
    ax = fig.add_subplot(gs[0, col])
    ax.imshow(mnist_x_train[idx], cmap='gray', interpolation='nearest')
    ax.set_title(
        f"'{mnist_y_train[idx]}'",
        fontsize=11, fontweight='bold',
        color='#00e5ff', pad=3
    )
    ax.axis('off')
    for spine in ax.spines.values():
        spine.set_edgecolor('#00e5ff')
        spine.set_linewidth(1.2)
        spine.set_visible(True)

fig.text(
    0.005, 0.78, "MNIST\n(28×28\ngray)",
    va='center', ha='left', fontsize=8.5,
    color='#00e5ff', fontweight='bold',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='#1a1a2e', edgecolor='#00e5ff', lw=1)
)

for col, idx in enumerate(cifar_indices):
    ax = fig.add_subplot(gs[1, col])
    ax.imshow(cifar_x_train[idx], interpolation='nearest')
    label = CIFAR10_CLASSES[int(cifar_y_train[idx])]
    ax.set_title(
        label,
        fontsize=7.5, fontweight='bold',
        color='#ff6f61', pad=3
    )
    ax.axis('off')
    for spine in ax.spines.values():
        spine.set_edgecolor('#ff6f61')
        spine.set_linewidth(1.2)
        spine.set_visible(True)

fig.text(
    0.005, 0.28, "CIFAR-10\n(32×32\nRGB)",
    va='center', ha='left', fontsize=8.5,
    color='#ff6f61', fontweight='bold',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='#1a1a2e', edgecolor='#ff6f61', lw=1)
)

plt.savefig(
    'dataset_samples.png',
    dpi=180,
    bbox_inches='tight',
    facecolor=fig.get_facecolor()
)
plt.show()
print("\nSaved → dataset_samples.png")

(a) Shapes
Dataset Train images Train labels Test images Test labels
MNIST(60000, 28, 28)(60000,)(10000, 28, 28)(10000,)
CIFAR-10(50000, 32, 32, 3)(50000, 1)(10000, 32, 32, 3)(10000, 1)
MNIST has no channel dimension (grayscale), CIFAR-10 has 3 (RGB) — this matters when you build your CNN input layers.

(b) dtype & range — Both are uint8, range [0, 255]. You must divide by 255.0 to normalize to [0.0, 1.0] before training.

(c) Balance — MNIST is very balanced, ~5,900–6,300 samples per digit class. Std deviation is well under 300, so no resampling is needed. The bar chart printed in the output makes this visually obvious.

In [ ]:
import numpy as np
from tensorflow.keras.datasets import mnist, cifar10
import tensorflow as tf

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

(mnist_x_train, mnist_y_train), (mnist_x_test, mnist_y_test)   = mnist.load_data()
(cifar_x_train, cifar_y_train), (cifar_x_test,  cifar_y_test)  = cifar10.load_data()

cifar_y_train = cifar_y_train.squeeze()
cifar_y_test  = cifar_y_test.squeeze()

def banner(title):
    print(f"  {title}")

def before_after(label, before_arr, after_arr, show_values=True):
    """Print shape + dtype + sample values before and after a step."""
    print(f"\n  ┌─ {label}")
    print(f"  │  BEFORE → shape: {before_arr.shape}  dtype: {before_arr.dtype}")
    if show_values:
        sample = before_arr.flat[:6]
        print(f"  │           sample values: {list(sample)}")
    print(f"  │  AFTER  → shape: {after_arr.shape}  dtype: {after_arr.dtype}")
    if show_values:
        sample = after_arr.flat[:6]
        print(f"  └─          sample values: {[round(float(v),4) for v in sample]}")
    else:
        print(f"  └─          sample values: {after_arr[0]}")  # show one one-hot row

def preprocess(images, labels, dataset_name="Dataset"):
    """
    Full CNN-ready preprocessing pipeline.

    Steps
    ─────
    1. Normalise pixel values → float32 in [0.0, 1.0]
       WHY: Neural network weights are initialised near zero.
            Raw uint8 values (0-255) produce huge activations,
            making gradients explode and training unstable.
            Dividing by 255.0 (float, not int) guarantees the
            result is float32, not integer-truncated.

    2. Reshape MNIST (N,28,28) → (N,28,28,1)
       WHY: Keras Conv2D expects 4-D input:
            (batch, height, width, channels).
            MNIST is grayscale so channels=1.
            CIFAR-10 already has shape (N,32,32,3) — skipped.

    3. One-hot encode labels → vectors of length 10
       WHY: Softmax output is a 10-element probability vector.
            Integer labels can't be compared to that directly.
            One-hot turns label '3' → [0,0,0,1,0,0,0,0,0,0]
            so categorical cross-entropy loss works correctly.

    Parameters
    ──────────
    images       : np.ndarray  raw uint8 pixel array
    labels       : np.ndarray  integer class labels
    dataset_name : str         label for console output

    Returns
    ───────
    images_out : np.ndarray  float32, normalised, channel dim added
    labels_out : np.ndarray  float32, one-hot encoded (N, 10)
    """

    banner(f"{dataset_name} — Preprocessing Pipeline")

    print("\n  ▶  STEP 1: Normalise pixel values to [0.0, 1.0]")
    print()
    print("     # Why 255.0 not 255?")
    print("     # 255   is int  → result dtype stays uint8  (WRONG)")
    print("     # 255.0 is float→ result dtype becomes float64,")
    print("     #                  then we cast to float32  (CORRECT)")

    images_norm = (images / 255.0).astype(np.float32)

    before_after("Normalisation", images, images_norm)

    # Sanity checks
    assert images_norm.dtype  == np.float32,  " dtype must be float32"
    assert images_norm.min()  >= 0.0,         " min below 0.0"
    assert images_norm.max()  <= 1.0,         " max above 1.0"
    print(f"\n   Range confirmed → min={images_norm.min():.4f}  "
          f"max={images_norm.max():.4f}  dtype={images_norm.dtype}")

    print("\n  ▶  STEP 2: Reshape — add channel dimension (MNIST only)")

    if images_norm.ndim == 3:
        images_reshaped = images_norm.reshape(
            images_norm.shape[0],
            images_norm.shape[1],
            images_norm.shape[2],
            1
        )
        before_after("Reshape", images_norm, images_reshaped, show_values=False)
        print(f"  Channel dim added: {images_norm.shape} → {images_reshaped.shape}")
    else:
        images_reshaped = images_norm
        print(f"\n  ⏭  Skipped — {dataset_name} already has channel dim: "
              f"{images_norm.shape}")

    print("\n  ▶  STEP 3: One-hot encode integer labels → length-10 vectors")
    print()
    print("     # Manual one-hot (no to_categorical utility):")
    print("     # 1. Create zero matrix of shape (N, 10)")
    print("     # 2. Use NumPy advanced indexing to set the")
    print("     #    correct column to 1 for each row.")

    num_classes = 10
    n           = labels.shape[0]

    labels_onehot              = np.zeros((n, num_classes), dtype=np.float32)
    labels_onehot[np.arange(n), labels.astype(int)] = 1.0

    print(f"\n  ┌─ One-Hot Encoding")
    print(f"  │  BEFORE → shape: {labels.shape}  dtype: {labels.dtype}")
    print(f"  │           sample (first 8 labels): {labels[:8].tolist()}")
    print(f"  │  AFTER  → shape: {labels_onehot.shape}  dtype: {labels_onehot.dtype}")
    print(f"  │           sample (first 3 one-hot rows):")
    for i in range(3):
        row = labels_onehot[i].astype(int).tolist()
        print(f"  │             label={labels[i]}  →  {row}")
    print(f"  └─")

    # Sanity checks
    assert labels_onehot.shape == (n, 10),           " shape wrong"
    assert labels_onehot.dtype == np.float32,        " dtype must be float32"
    assert np.all(labels_onehot.sum(axis=1) == 1.0), " each row must sum to 1"
    print(f"\n   One-hot verified — every row sums to 1.0  "
          f"dtype={labels_onehot.dtype}")

    return images_reshaped, labels_onehot

mnist_x_train_p, mnist_y_train_p = preprocess(mnist_x_train, mnist_y_train, "MNIST Train")
mnist_x_test_p,  mnist_y_test_p  = preprocess(mnist_x_test,  mnist_y_test,  "MNIST Test")

cifar_x_train_p, cifar_y_train_p = preprocess(cifar_x_train, cifar_y_train, "CIFAR-10 Train")
cifar_x_test_p,  cifar_y_test_p  = preprocess(cifar_x_test,  cifar_y_test,  "CIFAR-10 Test")

banner("FINAL SHAPES AFTER PREPROCESSING")
rows = [
    ("MNIST  train images",  mnist_x_train.shape,  mnist_x_train_p.shape,  "uint8", "float32"),
    ("MNIST  train labels",  mnist_y_train.shape,  mnist_y_train_p.shape,  "uint8", "float32"),
    ("MNIST  test  images",  mnist_x_test.shape,   mnist_x_test_p.shape,   "uint8", "float32"),
    ("MNIST  test  labels",  mnist_y_test.shape,   mnist_y_test_p.shape,   "uint8", "float32"),
    ("CIFAR  train images",  cifar_x_train.shape,  cifar_x_train_p.shape,  "uint8", "float32"),
    ("CIFAR  train labels",  cifar_y_train.shape,  cifar_y_train_p.shape,  "uint8", "float32"),
    ("CIFAR  test  images",  cifar_x_test.shape,   cifar_x_test_p.shape,   "uint8", "float32"),
    ("CIFAR  test  labels",  cifar_y_test.shape,   cifar_y_test_p.shape,   "uint8", "float32"),
]

print(f"\n  {'Array':<25} {'Before':>16}  {'After':>20}  {'dtype Before→After'}")
print("  " + "─" * 78)
for name, b_shape, a_shape, b_dt, a_dt in rows:
    print(f"  {name:<25} {str(b_shape):>16}  {str(a_shape):>20}  {b_dt} → {a_dt}")

print("\n  Preprocessing pipeline complete — all arrays are CNN-ready.")

In [ ]:
# TASK 4: Data Augmentation Pipeline

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.ndimage import rotate, zoom
import tensorflow as tf
import random, os

SEED = 42
random.seed(SEED); np.random.seed(SEED)
tf.random.set_seed(SEED); os.environ['PYTHONHASHSEED'] = str(SEED)

from tensorflow.keras.datasets import cifar10

(cifar_x_train, cifar_y_train), _ = cifar10.load_data()
cifar_y_train = cifar_y_train.squeeze()

CIFAR10_CLASSES = [
    'airplane','automobile','bird','cat','deer',
    'dog','frog','horse','ship','truck'
]

cifar_x_train = (cifar_x_train / 255.0).astype(np.float32)


def horizontal_flip(image, p=0.5):
    if random.random() < p:
        return np.fliplr(image)
    return image.copy()


def random_rotation(image, max_deg=10):
    angle = random.uniform(-max_deg, max_deg)
    rotated = rotate(
        image, angle,
        axes=(0, 1),
        reshape=False,
        mode='nearest',
        cval=0.0
    )
    return np.clip(rotated, 0.0, 1.0).astype(np.float32)


def random_zoom(image, max_zoom=0.10):
    factor = random.uniform(1.0, 1.0 + max_zoom)
    h, w, c = image.shape
    zoomed = zoom(image, (factor, factor, 1), order=1)
    zh, zw = zoomed.shape[:2]
    y0 = (zh - h) // 2
    x0 = (zw - w) // 2
    cropped = zoomed[y0:y0 + h, x0:x0 + w, :]
    return np.clip(cropped, 0.0, 1.0).astype(np.float32)


def augment(image, flip_p=0.5, max_rotation_deg=10, max_zoom_pct=0.10):
    # FIX 1: function body was empty — added the three transforms
    image = horizontal_flip(image, p=flip_p)
    image = random_rotation(image, max_deg=max_rotation_deg)
    image = random_zoom(image, max_zoom=max_zoom_pct)
    return image


N_IMAGES          = 5
AUGMENTS_PER_IMAGE = 3
sample_indices    = [0, 7, 14, 23, 31]

fig, axes = plt.subplots(
    N_IMAGES, AUGMENTS_PER_IMAGE + 1,
    figsize=(13, 17)
)
fig.patch.set_facecolor('#0d0d0d')

COL_LABELS = ["Original", "Augmented #1", "Augmented #2", "Augmented #3"]
COL_COLORS = ['#aaaaaa', '#00e5ff', '#00e5ff', '#00e5ff']

for col, (lbl, col_color) in enumerate(zip(COL_LABELS, COL_COLORS)):
    axes[0][col].set_title(lbl, fontsize=11, color=col_color,
                           fontweight='bold', pad=8)

for row, idx in enumerate(sample_indices):
    original   = cifar_x_train[idx]
    class_name = CIFAR10_CLASSES[cifar_y_train[idx]]

    images_in_row = [original] + [augment(original) for _ in range(AUGMENTS_PER_IMAGE)]

    for col, img in enumerate(images_in_row):
        ax = axes[row][col]
        ax.imshow(img, interpolation='nearest')
        ax.axis('off')

        border_color = '#555555' if col == 0 else '#00e5ff'
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor(border_color)
            spine.set_linewidth(2.2)

        if col == 0:
            ax.set_ylabel(
                f"{class_name.upper()}\n(idx {idx})",
                fontsize=9, color='#ffcc00', fontweight='bold',
                labelpad=6, rotation=0, ha='right', va='center'
            )

plt.subplots_adjust(hspace=0.08, wspace=0.06,
                    left=0.12, right=0.98,
                    top=0.97,  bottom=0.01)

plt.savefig('augmentation_demo.png', dpi=160,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print("Saved → augmentation_demo.png")  # FIX 2: removed extra )

QUES
Why must augmentation be applied only to the training set and never to the validation or
test set?

ANSWER
Validation/test sets measure real-world performance. They must represent the exact distribution of data the model will see in production — unmodified. If you flip or rotate test images, you're no longer measuring "does the model recognise this cat?" but "does it recognise a randomly distorted cat?", which is a different and meaningless question.
Augmentation is a regularisation trick, not a data transformation. Its entire purpose is to make the training distribution artificially harder and more varied so the model doesn't memorise. Applying it to validation would also corrupt your loss/accuracy signal — you'd get different metric values every epoch just from the random transforms, making it impossible to compare runs or detect overfitting reliably.
Label-preserving note: All three transforms here are safe for CIFAR-10. Horizontal flip is explicitly disabled for MNIST — a flipped 6 becomes a 9, breaking the label, which is a silent accuracy killer.

ANSWER1 The C (channel) dimension represents the number of independent measurements recorded at each pixel location.
For a greyscale image, C = 1 — each pixel stores a single intensity value (how bright that point is). Shape: (N, 28, 28, 1). There is only one "layer" of information per pixel.
For an RGB image, C = 3 — each pixel stores three separate values: Red intensity, Green intensity, and Blue intensity. Shape: (N, 32, 32, 3). The three channels stack depth-wise; together they encode colour. A Conv2D filter slides across H and W, but reaches through all C channels at once, which is why the channel count directly affects the number of learnable parameters in every convolutional layer.

ANSWER2
Technique 1 — Mini-batch loading with a data generator. Instead of loading all images into RAM at once, you yield small batches (e.g., 32 images) from disk on demand. At any moment only one batch occupies memory. Keras's ImageDataGenerator / tf.data pipelines do this natively. A full 1024×1024 RGB dataset at float32 is ~3 GB per 1,000 images — far too large to hold in GPU VRAM simultaneously.
Technique 2 — Prefetching and caching with tf.data. dataset.prefetch(tf.data.AUTOTUNE) lets the CPU prepare the next batch while the GPU trains on the current one, eliminating the idle wait. Combined with .cache() for a dataset that fits on disk but not RAM, this keeps the GPU saturated without ever materialising the full dataset in memory.

In [ ]:
ANSWER3
This is data leakage. Normalisation is supposed to shift and scale data relative to the training distribution — the statistics the model was built around. Computing mean and standard deviation from the test set instead means:

The test set is used to influence its own evaluation, making the model appear to perform better than it truly generalises.

In [ ]:
!git config --global user.email "tanyamittal158@gmail.com"
!git config --global user.name "tanyaamittal24"

In [ ]:
import os
files = ['dataset_samples.png']
for f in files:
    print(f"{'✅' if os.path.exists(f) else '❌'} {f}")

In [ ]:
%%bash
cd /content
ls -la *.png 2>/dev/null && echo "PNGs found" || echo "No PNGs"

In [ ]:
%%bash
cd /content

# Remove duplicate remote
git remote remove origin
git remote add origin https://github.com/tanyaamittal24/ml-assignment-cnn.git

# Add only what exists (no *.ipynb if it's missing)
git add dataset_samples.png
git add augmentation_demo.png 2>/dev/null || echo "augmentation_demo.png not found, skipping"

# If you want to save your notebook — Colab saves it to Drive, not /content
# So we add everything untracked instead
git add .

git status

In [ ]:
%%bash
cd /content
git commit -m "Add CNN lab Tasks 1-4: preprocessing, EDA, augmentation"

In [ ]:
%%bash
cd /content


In [ ]:
#TASK2
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import os

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


def conv2d(image, kernel, stride=1, padding=0):
    """
    Manual 2D convolution using only NumPy.
    No TensorFlow, no scipy, no np.convolve.

    Parameters
    ──────────
    image   : 2D np.ndarray  (H, W)
    kernel  : 2D np.ndarray  (K, K)
    stride  : int            step size for sliding window
    padding : int            zero-padding added to each side

    Returns
    ───────
    feature_map : 2D np.ndarray
    """
    H, W   = image.shape
    K      = kernel.shape[0]          # assume square kernel


    if padding > 0:
        image = np.pad(image,
                       pad_width=padding,
                       mode='constant',
                       constant_values=0)
        H, W = image.shape            # update after padding

    # Step 2 — Calculate output dimensions using formula:
    # out = floor((input + 2P - K) / S) + 1
    out_H = (H - K) // stride + 1
    out_W = (W - K) // stride + 1

    # Step 3 — Slide kernel across image
    feature_map = np.zeros((out_H, out_W), dtype=np.float64)

    for i in range(out_H):
        for j in range(out_W):
            row_start = i * stride
            col_start = j * stride
            # Extract patch same size as kernel
            patch = image[row_start : row_start + K,
                          col_start : col_start + K]
            # Element-wise multiply + sum = one conv output
            feature_map[i, j] = np.sum(patch * kernel)

    return feature_map

test_image = np.array([
    [3, 1, 0, 2, 4],
    [1, 5, 3, 2, 1],
    [0, 2, 6, 4, 3],
    [2, 3, 1, 5, 2],
    [1, 0, 2, 3, 4]
], dtype=np.float64)

sobel_x = np.array([
    [-1,  0,  1],
    [-2,  0,  2],
    [-1,  0,  1]
], dtype=np.float64)

result = conv2d(test_image, sobel_x, stride=1, padding=0)

print("  PROBLEM 1: Manual conv2d — Sobel-X on 5×5 image")
print(f"\n  Input image shape  : {test_image.shape}")
print(f"  Kernel shape       : {sobel_x.shape}")
print(f"  Stride             : 1   Padding : 0")
print(f"\n  Expected output shape formula:")
print(f"  out = floor((5 + 2×0 − 3) / 1) + 1 = 3")
print(f"  → Output shape: (3, 3)  Got: {result.shape}")
print(f"\n  Feature map output:")
print(result)


In [ ]:
def output_size(input_size, K, P, S):
    return (input_size + 2*P - K) // S + 1

print("\nPROBLEM 2: Output Size Derivations")

a = output_size(28, 5, 0, 1)
print(f"\n  (a) Input=28, K=5, P=0, S=1")
print(f"      (28 + 0 - 5) / 1 + 1 = {a}×{a}")

b = output_size(28, 3, 1, 1)
print(f"\n  (b) Input=28, K=3, P=1, S=1")
print(f"      (28 + 2 - 3) / 1 + 1 = {b}×{b}  (same padding preserves size)")

c = output_size(32, 3, 0, 2)
print(f"\n  (c) Input=32, K=3, P=0, S=2")
print(f"      (32 + 0 - 3) / 2 + 1 = {c}×{c}")

d1 = output_size(32, 3, 1, 1)
d2 = output_size(d1,  3, 0, 1)
print(f"\n  (d) Input=32, Layer1: K=3,P=1,S=1  →  Layer2: K=3,P=0,S=1")
print(f"      After Layer 1: (32 + 2 - 3)/1 + 1 = {d1}×{d1}")
print(f"      After Layer 2: (32 + 0 - 3)/1 + 1 = {d2}×{d2}")
print(f"      Final size: {d2}×{d2}")


In [ ]:
print("\nPROBLEM 3: LeNet-5 Implementation")

lenet5 = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(6,  5, padding='valid', activation='tanh', name='C1_conv'),
    layers.AveragePooling2D(2, strides=2,                    name='S2_avgpool'),
    layers.Conv2D(16, 5, padding='valid', activation='tanh', name='C3_conv'),
    layers.AveragePooling2D(2, strides=2,                    name='S4_avgpool'),
    layers.Flatten(                                          name='Flatten'),
    layers.Dense(120, activation='tanh',                     name='F5_dense'),
    layers.Dense(84,  activation='tanh',                     name='F6_dense'),
    layers.Dense(10,  activation='softmax',                  name='Output'),
], name='LeNet-5')

lenet5.summary()

# (b) Manual parameter count for first Conv2D layer
K, C_in, C_out = 5, 1, 6
params_c1 = (K * K * C_in + 1) * C_out
print(f"\n  (b) C1 layer parameter count:")
print(f"      Formula : (K×K×C_in + 1) × C_out")
print(f"              = ({K}×{K}×{C_in} + 1) × {C_out}")
print(f"              = {K*K*C_in + 1} × {C_out} = {params_c1}")

print("""
  (c) Why AvgPool in LeNet-5 vs MaxPool today?

  LeNet-5 used AvgPool because it fit the smooth mathematical
  framework of 1998 and paired well with Tanh activations.

  MaxPool dominates today because:
  - Retains the strongest feature per region — more translation invariant
  - Works better with ReLU (dominant since AlexNet 2012)
  - Empirically yields higher accuracy on modern benchmarks
""")

In [ ]:
print("PROBLEM 4: Custom CNN for CIFAR-10")

print("""
  Architecture Diagram:

  Input (32×32×3)
       │
  ┌────▼────────────────────────────────┐
  │ BLOCK 1                             │
  │  Conv2D(32, 3×3, same)              │
  │  BatchNorm → ReLU                   │
  │  Conv2D(32, 3×3, same)              │
  │  BatchNorm → ReLU                   │
  │  MaxPool(2×2) → Dropout(0.25)       │
  └────┬────────────────────────────────┘
       │ (16×16×32)
  ┌────▼────────────────────────────────┐
  │ BLOCK 2                             │
  │  Conv2D(64, 3×3, same)              │
  │  BatchNorm → ReLU                   │
  │  Conv2D(64, 3×3, same)              │
  │  BatchNorm → ReLU                   │
  │  MaxPool(2×2) → Dropout(0.25)       │
  └────┬────────────────────────────────┘
       │ (8×8×64)
  ┌────▼────────────────────────────────┐
  │ BLOCK 3                             │
  │  Conv2D(128, 3×3, same)             │
  │  BatchNorm → ReLU                   │
  │  Conv2D(128, 3×3, same)             │
  │  BatchNorm → ReLU                   │
  │  MaxPool(2×2) → Dropout(0.25)       │
  └────┬────────────────────────────────┘
       │ (4×4×128)
  GlobalAveragePooling2D → (128,)
  Dense(256) → BN → ReLU → Dropout(0.5)
  Dense(10, softmax)

  Design Rationale:
  - Filters double each block (32→64→128): early layers learn
    edges, deeper layers learn semantic features.
  - Double Conv before each pool (VGG-style) adds depth cheaply.
  - BatchNorm after every Conv stabilises and speeds up training.
  - GlobalAveragePooling cuts params drastically vs Flatten.
  - Dropout(0.25) + Dropout(0.5) prevents overfitting.
""")

def build_custom_cnn():
    model = keras.Sequential(name="CustomCNN_CIFAR10")
    model.add(layers.Input(shape=(32, 32, 3)))

    # Block 1
    model.add(layers.Conv2D(32, 3, padding='same', name='B1_conv1'))
    model.add(layers.BatchNormalization(name='B1_bn1'))
    model.add(layers.ReLU(name='B1_relu1'))
    model.add(layers.Conv2D(32, 3, padding='same', name='B1_conv2'))
    model.add(layers.BatchNormalization(name='B1_bn2'))
    model.add(layers.ReLU(name='B1_relu2'))
    model.add(layers.MaxPooling2D(2, name='B1_pool'))
    model.add(layers.Dropout(0.25, name='B1_drop'))

    # Block 2
    model.add(layers.Conv2D(64, 3, padding='same', name='B2_conv1'))
    model.add(layers.BatchNormalization(name='B2_bn1'))
    model.add(layers.ReLU(name='B2_relu1'))
    model.add(layers.Conv2D(64, 3, padding='same', name='B2_conv2'))
    model.add(layers.BatchNormalization(name='B2_bn2'))
    model.add(layers.ReLU(name='B2_relu2'))
    model.add(layers.MaxPooling2D(2, name='B2_pool'))
    model.add(layers.Dropout(0.25, name='B2_drop'))

    # Block 3
    model.add(layers.Conv2D(128, 3, padding='same', name='B3_conv1'))
    model.add(layers.BatchNormalization(name='B3_bn1'))
    model.add(layers.ReLU(name='B3_relu1'))
    model.add(layers.Conv2D(128, 3, padding='same', name='B3_conv2'))
    model.add(layers.BatchNormalization(name='B3_bn2'))
    model.add(layers.ReLU(name='B3_relu2'))
    model.add(layers.MaxPooling2D(2, name='B3_pool'))
    model.add(layers.Dropout(0.25, name='B3_drop'))

    # Classification Head
    model.add(layers.GlobalAveragePooling2D(name='GAP'))
    model.add(layers.Dense(256, name='FC1'))
    model.add(layers.BatchNormalization(name='FC1_bn'))
    model.add(layers.ReLU(name='FC1_relu'))
    model.add(layers.Dropout(0.5, name='FC1_drop'))
    model.add(layers.Dense(10, activation='softmax', name='Output'))

    return model

custom_cnn = build_custom_cnn()
custom_cnn.summary()

total_params = custom_cnn.count_params()
print(f"\n  Total parameters: {total_params:,}")
assert 200_000 <= total_params <= 2_000_000, \
    f"Params {total_params:,} outside 200K–2M range!"
print(f"  Within required 200K–2M range.")

ANSWER1 Two stacked 3×3 vs one 5×5 Conv — parameter comparison

  Assume C_in = C_out = 64 filters.

  One 5×5 Conv:
    (5×5×64 + 1) × 64 = 1601 × 64 = 102,464 params

  Two stacked 3×3 Convs:
    Layer 1: (3×3×64 + 1) × 64 = 577 × 64 = 36,928
    Layer 2: (3×3×64 + 1) × 64 = 577 × 64 = 36,928
    Total  :                               = 73,856 params

  Saving: 102,464 − 73,856 = 28,608 fewer params (~28% less)
  Two stacked 3×3 layers use FEWER parameters.

  Additional advantages:
  - Two ReLUs instead of one → more non-linearity, more expressive
  - Same receptive field (two 3×3 sees a 5×5 region)
  - Better regularisation due to more layers with fewer weights
  - Faster on GPU tensor cores optimised for small matrix shapes


ANSWER2
Role of Batch Normalisation

  BatchNorm normalises each mini-batch to mean=0, variance=1,
  then applies learnable scale γ and shift β per channel.
  This fights internal covariate shift — the drift in each
  layer's input distribution as weights update during training.

  Placement: Conv → BN → Activation (BEFORE activation).
  BN before ReLU ensures the activation sees a zero-centred
  input, maximising ReLU's linear region and preventing
  saturation. This is the dominant convention empirically.

  Two empirical benefits:
  1. Allows much higher learning rates — bounded activations
     prevent gradient explosion, so training is 5-10× faster.
  2. Acts as regularisation — batch statistics inject noise
     similar to Dropout, reducing overfitting without needing
     large dropout rates or heavy L2 penalties.

ANSWER3
GlobalAveragePooling vs Flatten

  GAP takes shape (H, W, C) and averages spatially per channel
  → output is (C,). It collapses the entire H×W grid to one
  number per channel.

  For our model after Block 3: shape is (4, 4, 128).
    GAP output     → (128,)    → Dense(10): 128×10+10 =  1,290 params
    Flatten output → (2048,)   → Dense(10): 2048×10+10 = 20,490 params

  If replaced with Flatten:
  - Parameter count grows 16× in the Dense head → more overfit risk
  - Loses translation invariance: Dense must learn which spatial
    position matters, requiring far more data to generalise
  - GAP doesn't care WHERE a feature fires, only WHETHER it fires
  - Flatten-based models don't transfer as well across input sizes

In [ ]:
# Python cell
from google.colab import drive
drive.mount('/drive')

import glob, shutil

notebooks = glob.glob('/drive/MyDrive/**/*.ipynb', recursive=True)
for i, nb in enumerate(notebooks):
    print(i, nb)

In [ ]:
%%bash
cd /content

git add cnn_assignment.ipynb dataset_samples.png

git status

git commit -m "Add Tasks 1-2: setup, EDA, preprocessing, augmentation, CNN"


In [ ]:
%%bash
ls /content/*.ipynb 2>/dev/null || echo "NO NOTEBOOK FOUND"
ls /content/*.png 2>/dev/null || echo "NO PNGs FOUND"

In [ ]:
# Python cell
from google.colab import drive
drive.mount('/drive')

import glob, shutil

notebooks = glob.glob('/drive/MyDrive/**/*.ipynb', recursive=True)
for i, nb in enumerate(notebooks):
    print(i, nb)

In [ ]:
# Python cell
shutil.copy(notebooks[0], '/content/cnn_assignment.ipynb')
print("✅ Done:", glob.glob('/content/*.ipynb'))

In [ ]:
%%bash
cd /content

git add cnn_assignment.ipynb dataset_samples.png augmentation_demo.png

git status

git commit -m "Add Tasks 1-2: setup, EDA, preprocessing, augmentation, CNN"


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.datasets import mnist, cifar10
import os

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)


# Preprocessing helper
def preprocess_mnist():
    (x_train, y_train), (x_test, y_test) = mnist.load_data()
    x_train = (x_train / 255.0).astype(np.float32).reshape(-1, 28, 28, 1)
    x_test  = (x_test  / 255.0).astype(np.float32).reshape(-1, 28, 28, 1)
    y_train = np.eye(10)[y_train].astype(np.float32)
    y_test  = np.eye(10)[y_test].astype(np.float32)
    return x_train, y_train, x_test, y_test

def preprocess_cifar():
    (x_train, y_train), (x_test, y_test) = cifar10.load_data()
    x_train = (x_train / 255.0).astype(np.float32)
    x_test  = (x_test  / 255.0).astype(np.float32)
    y_train = np.eye(10)[y_train.squeeze()].astype(np.float32)
    y_test  = np.eye(10)[y_test.squeeze()].astype(np.float32)
    return x_train, y_train, x_test, y_test


# LeNet-5 builder (reused from Task 2)
def build_lenet5():
    return keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(6,  5, padding='valid', activation='tanh', name='C1'),
        layers.AveragePooling2D(2, strides=2,                    name='S2'),
        layers.Conv2D(16, 5, padding='valid', activation='tanh', name='C3'),
        layers.AveragePooling2D(2, strides=2,                    name='S4'),
        layers.Flatten(),
        layers.Dense(120, activation='tanh',                     name='F5'),
        layers.Dense(84,  activation='tanh',                     name='F6'),
        layers.Dense(10,  activation='softmax',                  name='Output'),
    ], name='LeNet5')

In [ ]:
print("PROBLEM 1: First Training Run")

x_train, y_train, x_test, y_test = preprocess_mnist()

model_p1 = build_lenet5()
model_p1.compile(
    optimizer=keras.optimizers.SGD(learning_rate=0.01),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_p1 = model_p1.fit(
    x_train, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.10,
    verbose=1
)

test_loss, test_acc = model_p1.evaluate(x_test, y_test, verbose=0)
print(f"\n  Final Test Accuracy : {test_acc:.4f}")
print(f"  Final Test Loss     : {test_loss:.4f}")

# Detect overfitting epoch
val_losses = history_p1.history['val_loss']
overfit_epoch = None
for i in range(1, len(val_losses)):
    if val_losses[i] > val_losses[i-1]:
        overfit_epoch = i + 1
        break

if overfit_epoch:
    print(f"  First sign of overfitting at epoch: {overfit_epoch}")
else:
    print("  No clear overfitting detected in 15 epochs")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle('LeNet-5 on MNIST — SGD (lr=0.01, bs=64)',
             color='white', fontsize=13, fontweight='bold')

epochs_range = range(1, 16)

for ax in axes:
    ax.set_facecolor('#1a1a1a')
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#444')

# Loss plot
axes[0].plot(epochs_range, history_p1.history['loss'],
             color='#00e5ff', lw=2, label='Train Loss')
axes[0].plot(epochs_range, history_p1.history['val_loss'],
             color='#ff6f61', lw=2, label='Val Loss', linestyle='--')
if overfit_epoch:
    axes[0].axvline(overfit_epoch, color='yellow', linestyle=':',
                    lw=1.5, label=f'Overfit epoch {overfit_epoch}')
axes[0].set_title('Loss Curves', color='white', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(facecolor='#222', labelcolor='white')

# Accuracy plot
axes[1].plot(epochs_range, history_p1.history['accuracy'],
             color='#00e5ff', lw=2, label='Train Acc')
axes[1].plot(epochs_range, history_p1.history['val_accuracy'],
             color='#ff6f61', lw=2, label='Val Acc', linestyle='--')
if overfit_epoch:
    axes[1].axvline(overfit_epoch, color='yellow', linestyle=':',
                    lw=1.5, label=f'Overfit epoch {overfit_epoch}')
axes[1].set_title('Accuracy Curves', color='white', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(facecolor='#222', labelcolor='white')

plt.tight_layout()
plt.savefig('lenet_sgd_curves.png', dpi=150,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print("Saved → lenet_sgd_curves.png")


In [ ]:
print("\nPROBLEM 2: Optimiser Comparison")

optimisers = {
    'SGD (lr=0.01)':            keras.optimizers.SGD(learning_rate=0.01),
    'SGD+Momentum (lr=0.01)':   keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    'Adam (lr=0.001)':          keras.optimizers.Adam(learning_rate=0.001),
}

histories_p2 = {}

for name, opt in optimisers.items():
    print(f"\n  Training with {name} ...")
    tf.random.set_seed(SEED)
    m = build_lenet5()
    m.compile(optimizer=opt,
              loss='categorical_crossentropy',
              metrics=['accuracy'])
    h = m.fit(x_train, y_train,
              epochs=15, batch_size=64,
              validation_split=0.10,
              verbose=0)
    histories_p2[name] = h
    final_val = h.history['val_accuracy'][-1]
    print(f"  Final val accuracy: {final_val:.4f}")

# Plot all three val accuracy curves
colors = ['#00e5ff', '#7fff00', '#ff6f61']
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#0d0d0d')
ax.set_facecolor('#1a1a1a')
ax.tick_params(colors='white')
ax.xaxis.label.set_color('white')
ax.yaxis.label.set_color('white')
for spine in ax.spines.values():
    spine.set_edgecolor('#444')

for (name, h), color in zip(histories_p2.items(), colors):
    ax.plot(range(1, 16), h.history['val_accuracy'],
            label=name, color=color, lw=2)

ax.set_title('Optimiser Comparison — Val Accuracy (MNIST)',
             color='white', fontweight='bold', fontsize=12)
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Accuracy')
ax.legend(facecolor='#222', labelcolor='white')
plt.tight_layout()
plt.savefig('optimiser_comparison.png', dpi=150,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print("Saved → optimiser_comparison.png")


In [ ]:
print("PROBLEM 3: LR × Batch Size Grid Search on CIFAR-10")

x_train_c, y_train_c, x_test_c, y_test_c = preprocess_cifar()

# Custom CNN from Task 2 (simplified rebuild here)
def build_custom_cnn():
    m = keras.Sequential([
        layers.Input(shape=(32, 32, 3)),
        layers.Conv2D(32, 3, padding='same'), layers.BatchNormalization(),
        layers.ReLU(), layers.MaxPooling2D(2), layers.Dropout(0.25),
        layers.Conv2D(64, 3, padding='same'), layers.BatchNormalization(),
        layers.ReLU(), layers.MaxPooling2D(2), layers.Dropout(0.25),
        layers.GlobalAveragePooling2D(),
        layers.Dense(128), layers.ReLU(), layers.Dropout(0.5),
        layers.Dense(10, activation='softmax'),
    ], name='CustomCNN')
    return m

learning_rates = [0.1, 0.01, 0.001]
batch_sizes    = [32, 128]

# results[lr][bs] = final val accuracy
results = {}

for lr in learning_rates:
    results[lr] = {}
    for bs in batch_sizes:
        print(f"  LR={lr}, BS={bs} ...", end=' ')
        tf.random.set_seed(SEED)
        m = build_custom_cnn()
        m.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
        h = m.fit(x_train_c, y_train_c,
                  epochs=10, batch_size=bs,
                  validation_split=0.10,
                  verbose=0)
        val_acc = h.history['val_accuracy'][-1]
        results[lr][bs] = val_acc
        print(f"val_acc={val_acc:.4f}")

# Print 3×2 results table
print("\n  Results Table (rows=LR, cols=Batch Size):")
print(f"  {'LR':<10} {'BS=32':>10} {'BS=128':>10}")
print("  " + "-" * 32)
for lr in learning_rates:
    print(f"  {lr:<10} {results[lr][32]:>10.4f} {results[lr][128]:>10.4f}")

best_lr = max(learning_rates, key=lambda lr: max(results[lr].values()))
best_bs = max(batch_sizes,    key=lambda bs: max(results[lr][bs] for lr in learning_rates))
print(f"\n  Best combination: LR={best_lr}, BS={best_bs}")

In [ ]:
print("PROBLEM 4: Regularisation Experiment — 4 variants")

def build_2block_cnn(use_dropout=False, use_batchnorm=False):
    model = keras.Sequential(name='RegCNN')
    model.add(layers.Input(shape=(32, 32, 3)))

    for filters in [32, 64]:
        model.add(layers.Conv2D(filters, 3, padding='same', activation='relu'))
        if use_batchnorm:
            model.add(layers.BatchNormalization())
        model.add(layers.MaxPooling2D(2))
        if use_dropout:
            model.add(layers.Dropout(0.3))

    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dense(128, activation='relu'))
    if use_dropout:
        model.add(layers.Dropout(0.5))
    model.add(layers.Dense(10, activation='softmax'))
    return model

variants = {
    'No Regularisation':         build_2block_cnn(False, False),
    'Dropout only':              build_2block_cnn(True,  False),
    'BatchNorm only':            build_2block_cnn(False, True),
    'Dropout + BatchNorm':       build_2block_cnn(True,  True),
}

histories_p4 = {}
colors_p4    = ['#ff6f61', '#00e5ff', '#7fff00', '#ffcc00']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle('Regularisation Comparison — CIFAR-10 (20 epochs)',
             color='white', fontsize=13, fontweight='bold')
axes = axes.flatten()

gaps = {}

for i, (name, model) in enumerate(variants.items()):
    print(f"\n  Training: {name}")
    tf.random.set_seed(SEED)
    model.compile(optimizer=keras.optimizers.Adam(0.001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    h = model.fit(x_train_c, y_train_c,
                  epochs=20, batch_size=64,
                  validation_split=0.10,
                  verbose=0)
    histories_p4[name] = h

    train_acc_final = h.history['accuracy'][-1]
    val_acc_final   = h.history['val_accuracy'][-1]
    gap             = train_acc_final - val_acc_final
    gaps[name]      = gap
    print(f"  Train acc: {train_acc_final:.4f}  Val acc: {val_acc_final:.4f}  Gap: {gap:.4f}")

    ax = axes[i]
    ax.set_facecolor('#1a1a1a')
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#444')

    ep = range(1, 21)
    ax.plot(ep, h.history['accuracy'],     color=colors_p4[i], lw=2, label='Train')
    ax.plot(ep, h.history['val_accuracy'], color=colors_p4[i], lw=2,
            linestyle='--', alpha=0.6,     label='Val')
    ax.set_title(name, color='white', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(facecolor='#222', labelcolor='white')

plt.tight_layout()
plt.savefig('regularisation_comparison.png', dpi=150,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print("Saved → regularisation_comparison.png")

print("\n  Train-Val Accuracy Gap at Final Epoch:")
print(f"  {'Variant':<25} {'Gap':>8}")
print("  " + "-" * 35)
for name, gap in gaps.items():
    print(f"  {name:<25} {gap:>8.4f}")
best_variant = min(gaps, key=gaps.get)
print(f"\n  Best (smallest gap): {best_variant}")

In [ ]:

print("PROBLEM 5: LR Scheduling — ReduceLROnPlateau vs Cosine Annealing")

# Use best variant from Problem 4
def build_best_model():
    return build_2block_cnn(use_dropout=True, use_batchnorm=True)

schedulers = {
    'ReduceLROnPlateau': keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, verbose=0),
    'CosineAnnealing':   keras.callbacks.CosineDecayRestarts(
        initial_learning_rate=0.001, first_decay_steps=10)
        if hasattr(keras.callbacks, 'CosineDecayRestarts')
        else keras.callbacks.LearningRateScheduler(
            lambda epoch: 0.001 * (1 + np.cos(np.pi * epoch / 30)) / 2)
}

histories_p5 = {}
lr_histories  = {}

for name, callback in schedulers.items():
    print(f"\n  Training with {name} ...")
    tf.random.set_seed(SEED)
    m = build_best_model()
    m.compile(optimizer=keras.optimizers.Adam(0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

    lr_tracker = keras.callbacks.LambdaCallback(
        on_epoch_end=lambda epoch, logs:
            lr_histories.setdefault(name, []).append(
                float(m.optimizer.learning_rate)))

    h = m.fit(x_train_c, y_train_c,
              epochs=30, batch_size=64,
              validation_split=0.10,
              callbacks=[callback, lr_tracker],
              verbose=0)
    histories_p5[name] = h
    print(f"  Final val accuracy: {h.history['val_accuracy'][-1]:.4f}")

# Plot side by side
colors_s = ['#00e5ff', '#ff6f61']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle('LR Scheduling Comparison — CIFAR-10 (30 epochs)',
             color='white', fontsize=13, fontweight='bold')

for ax in axes.flatten():
    ax.set_facecolor('#1a1a1a')
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#444')

ep = range(1, 31)
for i, (name, h) in enumerate(histories_p5.items()):
    # LR curve
    axes[0][i].plot(ep, lr_histories.get(name, [0]*30),
                    color=colors_s[i], lw=2)
    axes[0][i].set_title(f'{name} — LR vs Epoch',
                         color='white', fontweight='bold')
    axes[0][i].set_xlabel('Epoch')
    axes[0][i].set_ylabel('Learning Rate')

    # Val accuracy curve
    axes[1][i].plot(ep, h.history['val_accuracy'],
                    color=colors_s[i], lw=2)
    axes[1][i].set_title(f'{name} — Val Accuracy',
                         color='white', fontweight='bold')
    axes[1][i].set_xlabel('Epoch')
    axes[1][i].set_ylabel('Val Accuracy')

plt.tight_layout()
plt.savefig('lr_schedule_comparison.png', dpi=150,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print("Saved → lr_schedule_comparison.png")

ANSWER1
The loss landscape is a high-dimensional surface with valleys
  (minima) and ridges. Gradient descent takes steps proportional
  to the LR in the direction of steepest descent.

  With LR=1.0, each step is enormous. Instead of sliding into
  the valley floor, the update overshoots to the opposite wall,
  which is even higher — then overshoots back again, even higher.
  This is called divergence: loss grows each step rather than
  shrinking. Mathematically, the update rule x = x - lr * grad
  requires lr < 2/L (where L is the Lipschitz constant of the
  gradient) for guaranteed convergence. LR=1.0 almost always
  violates this for deep networks.

  Oscillation happens at intermediate bad LRs: the step crosses
  the valley but doesn't escape — it bounces side to side,
  making slow or no progress.

ANSWER2
Typical pattern seen in the 3×2 table:
  - LR=0.1 often diverges or performs poorly — too large for Adam
  - LR=0.01 is stable but slower than 0.001
  - LR=0.001 with BS=32 usually achieves the best val accuracy
    because smaller batches provide noisier gradients that act
    as implicit regularisation, helping escape sharp minima.
  - LR=0.001 with BS=128 is faster per epoch but slightly lower
    final accuracy — larger batches converge to sharper minima
    that generalise slightly worse (sharp minima hypothesis).

  Hypothesis: Smaller batch + well-tuned LR generalises best
  because gradient noise prevents overfitting to training set.

ANSWER3
Dropout at inference — scaling correction

  During training with Dropout(p=0.5), each neuron is randomly
  zeroed with probability 0.5. This means on average only 50%
  of neurons are active, so the expected output magnitude is
  halved compared to a fully-connected forward pass.

  At inference, ALL neurons are active (no dropout). Without
  correction, outputs would be roughly 2× larger than during
  training, shifting the distribution the next layer expects.

  Correction: multiply surviving activations by (1-p) = 0.5
  during training (inverted dropout), so inference needs NO
  scaling change. Keras implements inverted dropout by default —
  it scales activations UP by 1/(1-p) during training, then
  uses raw weights at test time. Net effect: expected magnitude
  is identical in both phases.


ANSWER4
ReduceLROnPlateau vs Cosine Annealing

  (i) What triggers LR reduction:
    ReduceLROnPlateau: REACTIVE — monitors a metric (val_loss)
      and reduces LR only when it stops improving for `patience`
      epochs. LR stays flat until a plateau is detected.
    Cosine Annealing: SCHEDULED — LR decays deterministically
      following a cosine curve from initial LR to near-zero,
      regardless of what the model is doing.

  (ii) Shape of the LR curve:
    ReduceLROnPlateau: staircase — flat then sudden drops.
    Cosine Annealing: smooth S-curve downward (or repeated
      cycles with warm restarts).

  (iii) When each is better suited:
    ReduceLROnPlateau: when you don't know in advance how many
      epochs are needed; it adapts to the training dynamics.
      Good for exploratory training runs.
    Cosine Annealing: when you have a fixed training budget and
      want smooth, predictable LR decay. Often gives better
      final accuracy as the slow smooth decay lets the model
      settle precisely into a flat minimum.

In [ ]:
%%bash
cd /content
git add lenet_sgd_curves.png optimiser_comparison.png regularisation_comparison.png lr_schedule_comparison.png
git commit -m "Add Task 3: training curves, optimiser comparison, regularisation, LR scheduling"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import cifar10
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import os

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

CIFAR10_CLASSES = ['airplane','automobile','bird','cat','deer',
                   'dog','frog','horse','ship','truck']

# Load & preprocess CIFAR-10
(x_train, y_train_raw), (x_test, y_test_raw) = cifar10.load_data()
x_train = (x_train / 255.0).astype(np.float32)
x_test  = (x_test  / 255.0).astype(np.float32)
y_train_raw = y_train_raw.squeeze()
y_test_raw  = y_test_raw.squeeze()
y_train = np.eye(10)[y_train_raw].astype(np.float32)
y_test  = np.eye(10)[y_test_raw].astype(np.float32)


# Build & train the best model from Task 3 (Dropout + BN)
def build_model():
    model = keras.Sequential([
        layers.Input(shape=(32, 32, 3)),

        layers.Conv2D(32, 3, padding='same', name='conv1'),
        layers.BatchNormalization(), layers.ReLU(),
        layers.Conv2D(32, 3, padding='same', name='conv2'),
        layers.BatchNormalization(), layers.ReLU(),
        layers.MaxPooling2D(2), layers.Dropout(0.25),

        layers.Conv2D(64, 3, padding='same', name='conv3'),
        layers.BatchNormalization(), layers.ReLU(),
        layers.Conv2D(64, 3, padding='same', name='conv4'),
        layers.BatchNormalization(), layers.ReLU(),
        layers.MaxPooling2D(2), layers.Dropout(0.25),

        layers.Conv2D(128, 3, padding='same', name='conv5'),
        layers.BatchNormalization(), layers.ReLU(),
        layers.MaxPooling2D(2), layers.Dropout(0.25),

        layers.GlobalAveragePooling2D(name='gap'),
        layers.Dense(256), layers.ReLU(), layers.Dropout(0.5),
        layers.Dense(10, activation='softmax'),
    ], name='CIFAR_CNN')
    return model

model = build_model()
model.compile(optimizer=keras.optimizers.Adam(0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

print("Training model...")
history = model.fit(x_train, y_train,
                    epochs=20, batch_size=64,
                    validation_split=0.10,
                    verbose=1)

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\nTest Accuracy: {test_acc:.4f}")



In [ ]:
print("\nPROBLEM 1: Visualising Conv1 Filters")

conv1_layer  = model.get_layer('conv1')
filters, _   = conv1_layer.get_weights()   # shape (3, 3, 3, 32)
n_filters    = filters.shape[-1]           # 32

# Normalise each filter independently to [0, 1]
fig, axes = plt.subplots(4, 8, figsize=(16, 9))
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle('Conv1 Learned Filters (32 filters, 3×3×3)',
             color='white', fontsize=13, fontweight='bold')

for i, ax in enumerate(axes.flatten()):
    ax.set_facecolor('#0d0d0d')
    if i < n_filters:
        f = filters[:, :, :, i]            # (3, 3, 3)  — RGB filter
        # Normalise to [0, 1] for display
        f_min, f_max = f.min(), f.max()
        f_norm = (f - f_min) / (f_max - f_min + 1e-8)
        ax.imshow(f_norm, interpolation='nearest')
        ax.set_title(f'#{i}', fontsize=7, color='#aaaaaa', pad=2)
    ax.axis('off')

plt.tight_layout()
plt.savefig('conv1_filters.png', dpi=150,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print("Saved → conv1_filters.png")

print("""
  Filter Pattern Observations (b):
  - Filters #0–#5:  appear to detect oriented edges (horizontal,
    vertical, diagonal) — high contrast between opposite sides,
    similar to Sobel kernels from Task 2.
  - Filters #6–#12: show colour-opponent patterns (red vs green,
    blue vs yellow) — similar to Gabor filters tuned to colour.
  - Filters #13–#20: detect colour blobs or uniform regions —
    one channel dominates, others near zero.
  - Filters #21–#31: mixed texture detectors — diagonal stripes
    or checkerboard patterns that respond to high-frequency texture.
  - A few filters are near-uniform (possible dead filters from
    ReLU saturation before BatchNorm stabilised training).
""")


In [ ]:
print("PROBLEM 2: Intermediate Feature Maps")

# Pick one correctly classified test image
y_pred_all = np.argmax(model.predict(x_test, verbose=0), axis=1)
correct_idx = np.where(y_pred_all == y_test_raw)[0]

sample_idx   = correct_idx[0]
sample_img   = x_test[sample_idx:sample_idx+1]
sample_label = CIFAR10_CLASSES[y_test_raw[sample_idx]]

print(f"Using test image idx={sample_idx}, class='{sample_label}'")


# ✅ BUILD MODEL (fixes original error)
_ = model.predict(sample_img)


# ✅ Use correct layer names
first_conv_model = tf.keras.Model(
    inputs=model.inputs,
    outputs=model.get_layer('conv1').output
)

last_conv_model = tf.keras.Model(
    inputs=model.inputs,
    outputs=model.get_layer('conv5').output
)


# Extract feature maps
fmaps_first = first_conv_model.predict(sample_img, verbose=0)[0]
fmaps_last  = last_conv_model.predict(sample_img, verbose=0)[0]


# Plot function
def plot_fmaps(fmaps, title, filename, n=8):
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))

    for i, ax in enumerate(axes.flatten()):
        if i < n:
            fm = fmaps[:, :, i]
            fm_norm = (fm - fm.min()) / (fm.max() - fm.min() + 1e-8)
            ax.imshow(fm_norm, cmap='viridis')
            ax.set_title(f'Channel {i}', fontsize=8)
        ax.axis('off')

    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(filename)
    plt.show()

    print(f"Saved → {filename}")


# Save outputs
plot_fmaps(
    fmaps_first,
    f'Feature Maps — Conv1 | Image: {sample_label}',
    'fmaps_layer1.png'
)

plot_fmaps(
    fmaps_last,
    f'Feature Maps — Conv5 | Image: {sample_label}',
    'fmaps_last.png'
)

In [ ]:
print("PROBLEM 3: Grad-CAM Heatmap")

import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt


# 🔴 STEP 0: BUILD MODEL (THIS FIXES ALL YOUR ERRORS)
_ = model.predict(x_test[:1])


grad_model = tf.keras.models.Model(
    inputs=model.inputs,
    outputs=[
        model.get_layer('conv5').output,
        model.layers[-1].output   # safer than model.output
    ]
)


def compute_gradcam(img, class_index):
    img_tensor = tf.convert_to_tensor(img)

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_tensor)

        # class-specific loss (IMPORTANT)
        loss = predictions[:, class_index]

    # gradients
    grads = tape.gradient(loss, conv_outputs)

    # global average pooling
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]

    # weighted sum
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)

    # ReLU
    heatmap = tf.maximum(heatmap, 0)

    # normalize
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)

    return heatmap.numpy()


def overlay_heatmap(heatmap, image, alpha=0.4):
    heatmap = cv2.resize(heatmap, (32, 32))

    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    image = (image[0] * 255).astype(np.uint8)

    superimposed = cv2.addWeighted(image, 1-alpha, heatmap, alpha, 0)
    return superimposed


y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)

correct_idx = np.where(y_pred == y_test_raw)[0]
wrong_idx   = np.where(y_pred != y_test_raw)[0]

correct_samples = correct_idx[:3]
wrong_sample    = wrong_idx[0]


fig, axes = plt.subplots(4, 3, figsize=(10, 12))

for i, idx in enumerate(list(correct_samples) + [wrong_sample]):

    img = x_test[idx:idx+1]
    true_label = y_test_raw[idx]
    pred_label = y_pred[idx]

    # Grad-CAM for TRUE class
    heatmap_true = compute_gradcam(img, true_label)
    overlay_true = overlay_heatmap(heatmap_true, img)

    # Grad-CAM for PREDICTED class
    heatmap_pred = compute_gradcam(img, pred_label)
    overlay_pred = overlay_heatmap(heatmap_pred, img)

    # Original image
    axes[i, 0].imshow(img[0])
    axes[i, 0].set_title(f"Original\nTrue: {CIFAR10_CLASSES[true_label]}")
    axes[i, 0].axis('off')

    # True class heatmap
    axes[i, 1].imshow(overlay_true)
    axes[i, 1].set_title("Grad-CAM (True)")
    axes[i, 1].axis('off')

    # Predicted class heatmap
    axes[i, 2].imshow(overlay_pred)
    axes[i, 2].set_title(f"Pred: {CIFAR10_CLASSES[pred_label]}")
    axes[i, 2].axis('off')


plt.tight_layout()
plt.savefig("gradcam_results.png")
plt.show()

print("Saved → gradcam_results.png")

For correctly classified images, the Grad-CAM heatmap highlights the main object region, showing the model focuses on relevant features.

For the misclassified image, the heatmap highlights background or irrelevant regions, indicating the model relies on incorrect features, leading to wrong prediction.

In [ ]:
print("PROBLEM 4: Confusion Matrix & Error Analysis")

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report


y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
y_true = y_test_raw


cm = confusion_matrix(y_true, y_pred)


# Plot confusion matrix
plt.figure(figsize=(10, 8))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.colorbar()

plt.xticks(range(10), CIFAR10_CLASSES, rotation=45)
plt.yticks(range(10), CIFAR10_CLASSES)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")

plt.tight_layout()
plt.savefig("confusion_matrix.png")
plt.show()

print("Saved → confusion_matrix.png")
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=CIFAR10_CLASSES))


cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)

# get top confusion pair
i, j = np.unravel_index(np.argmax(cm_no_diag), cm_no_diag.shape)

print("\nMost Confused Classes:")
print(f"True: {CIFAR10_CLASSES[i]}  → Predicted: {CIFAR10_CLASSES[j]}")
print(f"Count: {cm[i, j]}")


mis_idx = np.where(y_pred != y_true)[0][:5]

plt.figure(figsize=(12, 5))

for i, idx in enumerate(mis_idx):
    plt.subplot(1, 5, i+1)
    plt.imshow(x_test[idx])
    plt.title(
        f"T: {CIFAR10_CLASSES[y_true[idx]]}\nP: {CIFAR10_CLASSES[y_pred[idx]]}",
        fontsize=8
    )
    plt.axis('off')

plt.tight_layout()
plt.savefig("misclassified_examples.png")
plt.show()

print("Saved → misclassified_examples.png")

ANSWER1 The most confused classes are typically visually similar categories such as cat vs dog, deer vs horse, or automobile vs truck, as they share similar shapes and textures.

ANSWER2 Confusion occurs because different classes share similar visual features such as color, shape, and background. Low image resolution (32×32) further makes fine details hard to distinguish.

ANSWER3
Performance can be improved by:
- Using deeper architectures (ResNet, VGG)
- Applying stronger data augmentation
- Increasing training data
- Using higher resolution images
- Fine-tuning hyperparameters

In [ ]:
import glob

files = glob.glob('/content/drive/MyDrive/**/*.ipynb', recursive=True)

for f in files:
    print(f)

In [ ]:
import shutil

shutil.copy('/content/drive/MyDrive/Colab Notebooks/ML ASSIGNMENT.ipynb', '/content/')

In [ ]:
%%bash
cd /content
echo "drive/" >> .gitignore

In [ ]:
%%bash
cd /content
git add .

In [ ]:
%%bash
cd /content
git commit -m "task4 submission: CNN assignment"

In [ ]:
%%bash
cd /content

In [ ]:
%%bash
cd /content
git reset --soft HEAD~1

In [ ]:
%%bash
cd /content
git add .

In [ ]:
%%bash
cd /content
git commit -m "Final clean commit"

In [ ]:
%%bash
cd /content
git reset --soft $(git rev-list --max-parents=0 HEAD)

In [ ]:
%%bash
cd /content
git add .
git commit -m "Clean final submission"

In [ ]:
%%bash
cd /content